# Sovereign default onset

Predict whether reported conventional sovereign debt in default turns positive in the following calendar year. The eligible cohort has explicit prior-year zero debt in default and enough economic observations. The retrospective experiment uses revised WDI data and an assumed publication lag.

Run from a clone of the project on an Azure ML or local Python kernel with `pandas`, `scikit-learn`, `joblib`, and `openpyxl` installed. Supply the raw archive from `python -m ingestion.cli --all-countries --start-year 1990 --end-year 2025` and the [2026 BoC–BoE workbook](https://www.bankofcanada.ca/2026/10/staff-analytical-paper-2026-48/). You can set the paths below or pass them as environment variables.

In [ ]:
import os
import sys
from pathlib import Path

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo))

from ml.default_training import save_run, train

snapshot = Path(os.environ.get("WDI_SNAPSHOT", repo / "data/raw/global-wdi.json"))
workbook = Path(os.environ.get(
    "DEFAULT_WORKBOOK",
    repo / "data/raw/global-default/BoC-BoE-Database-2026.xlsx",
))
output = Path(os.environ.get("DEFAULT_MODEL_OUTPUT", repo / "ml/artifacts/default-onset"))

for path in (snapshot, workbook):
    if not path.is_file():
        raise FileNotFoundError(f"Set the input path to a downloaded or imported source: {path}")

Train through 2013 and choose the model and classification threshold using the 2014–2018 validation outcome years. Evaluate once on 2019–2025. Compare against the earlier observed event rate, which gives every row the same probability.

In [ ]:
import pandas as pd

report, model, predictions = train(snapshot, workbook, train_through=2013, validate_through=2018)
print(f"Selected model: {report['selected_model']}")
pd.DataFrame(report["splits"]).T

In [ ]:
pd.DataFrame(report["test"]).T

Save metrics, the research artifact, and predictions for the historical test period for error analysis. The dashboard does not load the research artifact. A vintage-aware evaluation is needed before forecasts can be interpreted for lending decisions.

In [ ]:
save_run(output, report, model, predictions)
predictions.sort_values("model_probability", ascending=False).head(10)